# 08. メタデータテーブル（Spark）

Iceberg のテーブルは、データファイルと、それを束ねるメタデータの階層でできています。

```
メタデータファイル（xxx.metadata.json）  … スキーマ、パーティション仕様、スナップショットの一覧
 └─ マニフェストリスト（snap-xxx.avro） … スナップショットごとに1つ。そのスナップショットのマニフェストの一覧
     └─ マニフェスト（xxx-m0.avro）     … データファイルの一覧と、列ごとの統計（最小値・最大値など）
         └─ データファイル（xxx.parquet）/ 削除ファイル
```

この階層は **メタデータテーブル** として SQL で覗けます。Spark では `<テーブル名>.<メタデータテーブル名>` と書きます。

- テーブル: `handson.meta_spark`

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("08_metadata_tables").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 準備: 観察用のテーブルを作る

`vendor` でパーティションを切り、INSERT を 2 回、DELETE を 1 回行います。
DELETE は、ファイルを書き直す代わりに「どの行を消したか」を記録した **削除ファイル** を書く方式（merge-on-read）にしておきます。

In [ ]:
sql("DROP TABLE IF EXISTS handson.meta_spark PURGE")
sql("""
CREATE TABLE handson.meta_spark (trip_id BIGINT, vendor STRING, fare DECIMAL(10, 2))
USING iceberg
PARTITIONED BY (vendor)
TBLPROPERTIES ('write.delete.mode' = 'merge-on-read')
""")
sql("INSERT INTO handson.meta_spark VALUES (1, 'A', 12.50), (2, 'B', 30.00), (3, 'A', 8.00)")
sql("INSERT INTO handson.meta_spark VALUES (4, 'C', 22.00), (5, 'A', 15.00)")
sql("DELETE FROM handson.meta_spark WHERE trip_id = 3")

## 1. metadata_log_entries: メタデータファイルの履歴

テーブルを変更するたびに、新しいメタデータファイル（`xxx.metadata.json`）が作られます。
カタログ（Polaris）が覚えているのは「今の最新のメタデータファイルはどれか」だけです。

In [ ]:
sql("SELECT timestamp, file, latest_snapshot_id FROM handson.meta_spark.metadata_log_entries ORDER BY timestamp")

## 2. snapshots / history: スナップショット

`manifest_list` 列が、そのスナップショットのマニフェストリストのファイルです。
`summary` には、追加・削除したファイル数や行数が記録されています。

In [ ]:
sql("""
SELECT committed_at, snapshot_id, operation, manifest_list,
       summary['added-records'] AS added_records,
       summary['added-delete-files'] AS added_delete_files
FROM handson.meta_spark.snapshots ORDER BY committed_at
""")
sql("SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor FROM handson.meta_spark.history")

## 3. manifests: マニフェスト

現在のスナップショットが参照しているマニフェストです。
`content` が 0 ならデータファイルの、1 なら削除ファイルのマニフェストです。

In [ ]:
sql("""
SELECT path, content, added_data_files_count, existing_data_files_count, added_delete_files_count,
       partition_summaries
FROM handson.meta_spark.manifests
""")

`partition_summaries` には、マニフェストに含まれるパーティションの値の範囲が入っています。
クエリの条件と範囲が重ならないマニフェストは、中身を読まずに飛ばせます。

## 4. files: データファイルと削除ファイル

`files` はデータファイルと削除ファイルの両方、`data_files` / `delete_files` はそれぞれだけを返します。
列ごとの統計（`lower_bounds` / `upper_bounds`）が、ファイル単位の読み飛ばしに使われます。

In [ ]:
sql("""
SELECT content, partition, record_count, file_size_in_bytes,
       readable_metrics.fare.lower_bound AS fare_min,
       readable_metrics.fare.upper_bound AS fare_max
FROM handson.meta_spark.files
ORDER BY content, partition.vendor
""")

`content = 1` の行が、DELETE で書かれた削除ファイル（位置削除: どのファイルの何行目を消したか）です。
中身を見てみます。

In [ ]:
path = spark.sql("SELECT file_path FROM handson.meta_spark.delete_files").first()[0]
print(path)
sql("SELECT file_path AS data_file, pos AS deleted_row_position FROM handson.meta_spark.position_deletes")

## 5. partitions: パーティションごとの集計

In [ ]:
sql("""
SELECT partition, record_count, file_count, position_delete_record_count, last_updated_at
FROM handson.meta_spark.partitions ORDER BY partition.vendor
""")

`record_count` はデータファイルの行数なので、削除ファイルで消した行も含みます。
実際に見える行数は、削除を適用した後の `SELECT count(*)` で数えます。

In [ ]:
sql("SELECT vendor, count(*) AS visible_rows FROM handson.meta_spark GROUP BY vendor ORDER BY vendor")

## 6. entries と all_*: 過去のスナップショットも含めて見る

- `entries`: マニフェストの各行（ファイルが追加されたのか、既存なのか、削除されたのか）
- `all_data_files` / `all_manifests`: 現在だけでなく、残っているすべてのスナップショットが参照するもの

07 の `expire_snapshots` で消えるのは、`all_data_files` にだけあって `data_files` にないファイルです（今は古いスナップショットも同じファイルを参照しているので、数は同じ）。

In [ ]:
sql("SELECT status, snapshot_id, data_file.content, data_file.record_count FROM handson.meta_spark.entries")
sql("""
SELECT
  (SELECT count(*) FROM handson.meta_spark.data_files)     AS current_data_files,
  (SELECT count(*) FROM handson.meta_spark.all_data_files) AS all_data_files,
  (SELECT count(*) FROM handson.meta_spark.manifests)      AS current_manifests,
  (SELECT count(*) FROM handson.meta_spark.all_manifests)  AS all_manifests
""")

`entries` の `status` は 0 = 既存、1 = 追加、2 = 削除 です。

## まとめ

- Iceberg のテーブルは「メタデータファイル → マニフェストリスト → マニフェスト → データファイル」の階層でできている
- メタデータテーブルで、その階層を SQL で覗ける
- マニフェストのパーティションの範囲と、ファイルごとの列の統計が、読み飛ばし（プルーニング）に使われる
- 同じ情報は Trino からも `"テーブル名$snapshots"` の形で見られる（trino.sql）